# 01 – Data Collection with the SpaceX REST API

**Goal:** collect past Falcon 9 launches and build one clean table (`data/dataset_part_1.csv`).

**Steps**
1. Request `https://api.spacexdata.com/v4/launches/past`.
2. Look up extra details from the `rockets`, `launchpads`, `payloads` and `cores` endpoints.
3. Keep only Falcon 9 launches, build one DataFrame, fill missing payload mass with the mean, save to CSV.

> **Note about the API (October 2026):** the live server `api.spacexdata.com` returned Cloudflare errors 522/525 on every endpoint when this project was built.
> The code below **tries the live API first**. If it fails, it uses saved copies of the **same API responses**:
> * launches: the official static copy published by the IBM course (`API_call_spacex_api.json`, the lab's recommended file),
> * rockets / payloads / launchpads / cores: archived copies of the real API responses from the Internet Archive (web.archive.org).
>
> No values are typed by hand. Everything comes from these JSON files.

In [1]:
import os, json, datetime
import requests
import pandas as pd
import numpy as np

RAW = "../data/raw"           # downloaded JSON files are kept here
os.makedirs(RAW, exist_ok=True)
API = "https://api.spacexdata.com/v4/"
IBM = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/"
WB = "https://web.archive.org/web/"
HEADERS = {"User-Agent": "spacex-capstone-student/1.0"}

## 1. Helper: try the live API, otherwise use the saved copy

In [2]:
def get_backup(backup_url, backup_file):
    """Download a saved JSON copy once (into data/raw) and read it."""
    path = os.path.join(RAW, backup_file)
    if not os.path.exists(path):
        r = requests.get(backup_url, headers=HEADERS, timeout=120)
        r.raise_for_status()
        open(path, "wb").write(r.content)
    return json.load(open(path, encoding="utf-8"))


def get_json(live_url, backup_url, backup_file):
    """Return JSON from the live API. If the API is down, use the backup copy."""
    try:
        r = requests.get(live_url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        print("LIVE  ", live_url)
        return r.json(), "live"
    except Exception as e:
        print("BACKUP", live_url, "->", type(e).__name__, str(e)[:60])
    return get_backup(backup_url, backup_file), "backup"

## 2. Request past launches

In [3]:
launches_json, src = get_json(API + "launches/past",
                              IBM + "API_call_spacex_api.json", "API_call_spacex_api.json")
data = pd.json_normalize(launches_json)
print("source:", src, "| rows:", data.shape[0], "| columns:", data.shape[1])
data[["flight_number", "name", "date_utc", "rocket", "launchpad"]].head()

BACKUP https://api.spacexdata.com/v4/launches/past -> ReadTimeout HTTPSConnectionPool(host='api.spacexdata.com', port=443): Re
source: backup | rows: 107 | columns: 42


,flight_number,name,date_utc,rocket,launchpad
0,1,FalconSat,2006-03-24T22:30:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86
1,2,DemoSat,2007-03-21T01:10:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86
2,3,Trailblazer,2008-08-03T03:34:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86
3,4,RatSat,2008-09-28T23:15:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86
4,5,RazakSat,2009-07-13T03:35:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86


## 3. Keep only the columns we need
Like the IBM lab: keep launches with **one core** and **one payload**, and launches up to 13 Nov 2020.

In [4]:
data = data[["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]]
data = data[data["cores"].map(len) == 1]          # remove Falcon Heavy (3 cores)
data = data[data["payloads"].map(len) == 1]       # remove launches with several payloads
data["cores"] = data["cores"].map(lambda x: x[0])
data["payloads"] = data["payloads"].map(lambda x: x[0])
data["date"] = pd.to_datetime(data["date_utc"]).dt.date
data = data[data["date"] <= datetime.date(2020, 11, 13)]
print("rows after filter:", len(data))

rows after filter: 94


## 4. Look-up tables from the other endpoints

In [5]:
# Rockets: id -> name
rockets_json, _ = get_json(API + "rockets", WB + "20230408043839id_/" + API + "rockets", "wb_v4_rockets.json")
rocket_name = {r["id"]: r["name"] for r in rockets_json}

# Payloads: id -> (mass in kg, orbit)
payloads_json, _ = get_json(API + "payloads", WB + "20240802054003id_/" + API + "payloads", "wb_v4_payloads.json")
payload_info = {p["id"]: (p["mass_kg"], p["orbit"]) for p in payloads_json}

print(len(rocket_name), "rockets,", len(payload_info), "payloads")

BACKUP https://api.spacexdata.com/v4/rockets -> HTTPError 522 Server Error: <none> for url: https://api.spacexdata.com


BACKUP https://api.spacexdata.com/v4/payloads -> ReadTimeout HTTPSConnectionPool(host='api.spacexdata.com', port=443): Re
4 rockets, 225 payloads


In [6]:
# Launchpads: id -> (name, longitude, latitude)
try:
    r = requests.get(API + "launchpads", headers=HEADERS, timeout=20); r.raise_for_status()
    pad_info = {p["id"]: (p["name"], p["longitude"], p["latitude"]) for p in r.json()}
    print("LIVE launchpads")
except Exception as e:
    print("BACKUP launchpads ->", type(e).__name__)
    pad_info = {}
    # archived v4 records for two pads
    for pid in ["5e9e4502f509092b78566f87", "5e9e4502f509094188566f88"]:
        p = get_backup(WB + "20220930191107id_/" + API + "launchpads/" + pid, f"wb_v4_launchpad_{pid}.json")
        pad_info[p["id"]] = (p["name"], p["longitude"], p["latitude"])
    # the third pad (Cape Canaveral SLC 40) only exists in the archived v3 API
    v3 = get_backup(WB + "20181127024739id_/https://api.spacexdata.com/v3/launchpads", "wb_v3_launchpads.json")
    slc40 = [p for p in v3 if p["site_id"] == "ccafs_slc_40"][0]
    pad_info["5e9e4501f509094ba4566f84"] = ("CCAFS SLC 40", slc40["location"]["longitude"], slc40["location"]["latitude"])
pad_info

BACKUP launchpads -> HTTPError


{'5e9e4502f509092b78566f87': ('VAFB SLC 4E', -120.610829, 34.632093),
 '5e9e4502f509094188566f88': ('KSC LC 39A', -80.6039558, 28.6080585),
 '5e9e4501f509094ba4566f84': ('CCAFS SLC 40', -80.577366, 28.5618571)}

In [7]:
# Cores: block, reuse count and serial number of each booster
try:
    r = requests.get(API + "cores", headers=HEADERS, timeout=20); r.raise_for_status()
    core_by_id = {c["id"]: (c["block"], c["reuse_count"], c["serial"]) for c in r.json()}
    core_by_flight = {}
    print("LIVE cores")
except Exception as e:
    print("BACKUP cores ->", type(e).__name__)
    core_by_id = {}
    # archived v3 cores list: each core lists the flight numbers it flew
    v3_cores = get_backup(WB +"20230602225826id_/https://api.spacexdata.com/v3/cores", "wb_v3_cores.json")
    core_by_flight = {}
    for c in v3_cores:
        for m in c["missions"]:
            core_by_flight[m["flight"]] = (c["block"], c["reuse_count"], c["core_serial"])
print(len(core_by_id) or len(core_by_flight), "core records")

BACKUP cores -> ReadTimeout
109 core records


## 5. Build one row per launch

In [8]:
rows = []
for _, r in data.iterrows():
    core = r["cores"]                                    # landing details are inside the launch record
    mass, orbit = payload_info[r["payloads"]]
    site, lon, lat = pad_info.get(r["launchpad"], (None, None, None))
    block, reuse_count, serial = core_by_id.get(core["core"]) or core_by_flight.get(r["flight_number"], (None, None, None))
    rows.append({
        "FlightNumber": r["flight_number"], "Date": r["date"],
        "BoosterVersion": rocket_name[r["rocket"]],
        "PayloadMass": mass, "Orbit": orbit, "LaunchSite": site,
        "Outcome": f"{core['landing_success']} {core['landing_type']}",
        "Flights": core["flight"], "GridFins": core["gridfins"], "Reused": core["reused"],
        "Legs": core["legs"], "LandingPad": core["landpad"],
        "Block": block, "ReusedCount": reuse_count, "Serial": serial,
        "Longitude": lon, "Latitude": lat})
launch_df = pd.DataFrame(rows)
launch_df["BoosterVersion"].value_counts()

BoosterVersion
Falcon 9    90
Falcon 1     4
Name: count, dtype: int64

## 6. Keep Falcon 9 only and renumber the flights

In [9]:
data_falcon9 = launch_df[launch_df["BoosterVersion"] == "Falcon 9"].reset_index(drop=True)
data_falcon9["FlightNumber"] = range(1, len(data_falcon9) + 1)
print("Falcon 9 launches:", len(data_falcon9))
data_falcon9.head()

Falcon 9 launches: 90


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,NaN,LEO,CCAFS SLC 40,None None,1,False,False,False,None,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.0,LEO,CCAFS SLC 40,None None,1,False,False,False,None,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.0,ISS,CCAFS SLC 40,None None,1,False,False,False,None,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.0,PO,VAFB SLC 4E,False Ocean,1,False,False,False,None,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.0,GTO,CCAFS SLC 40,None None,1,False,False,False,None,1.0,0,B1004,-80.577366,28.561857


## 7. Missing values

In [10]:
print(data_falcon9.isnull().sum())

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        5
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64


In [11]:
# PayloadMass: replace missing values with the mean. LandingPad: keep None (= no landing pad used).
mean_mass = data_falcon9["PayloadMass"].mean()
print("mean payload mass used to fill:", round(mean_mass, 2), "kg")
data_falcon9["PayloadMass"] = data_falcon9["PayloadMass"].fillna(mean_mass)
print("missing PayloadMass now:", data_falcon9["PayloadMass"].isnull().sum())

mean payload mass used to fill: 6123.55 kg
missing PayloadMass now: 0


## 8. Save

In [12]:
data_falcon9.to_csv("../data/dataset_part_1.csv", index=False)
print("saved ../data/dataset_part_1.csv", data_falcon9.shape)

saved ../data/dataset_part_1.csv (90, 17)


## 9. Sanity check against the IBM reference file
The IBM course publishes its own `dataset_part_1.csv`. We compare our result with it, column by column.

In [13]:
ibm = pd.read_csv(IBM + "dataset_part_1.csv")
ours = pd.read_csv("../data/dataset_part_1.csv")
print("shape ours:", ours.shape, "| IBM:", ibm.shape)
for col in ibm.columns:
    a, b = ours[col], ibm[col]
    if a.dtype.kind in "fi" and b.dtype.kind in "fi":
        same = np.isclose(a, b, equal_nan=True)
    else:
        same = a.astype(str) == b.astype(str)
    print(f"{col:15s} rows that differ: {(~same).sum()}")

shape ours: (90, 17) | IBM: (90, 17)
FlightNumber    rows that differ: 0
Date            rows that differ: 0
BoosterVersion  rows that differ: 0
PayloadMass     rows that differ: 16
Orbit           rows that differ: 0
LaunchSite      rows that differ: 0
Outcome         rows that differ: 0
Flights         rows that differ: 0
GridFins        rows that differ: 0
Reused          rows that differ: 0
Legs            rows that differ: 0
LandingPad      rows that differ: 0
Block           rows that differ: 0
ReusedCount     rows that differ: 6
Serial          rows that differ: 0
Longitude       rows that differ: 0
Latitude        rows that differ: 0


**Result:** almost all columns match the IBM file exactly. A few `PayloadMass` and `ReusedCount` values differ
because the archived API copy is newer than the one IBM used (SpaceX updated some payload masses and reuse counts later).
We keep **our own** values in the rest of the project.